In [3]:
from pathlib import Path
from urllib.request import urlopen
import shutil

def download_pretrained(option):
    """
    You can use this to download GloVe (or similar) model using wget.
    It's fine to download it through other means, but don't unzip the file (the rest of the notebook can
    handle the file in its zipped format).

    This function returns the path which you can use for the loader class that follows.

    The function will skip the download if it detects the file in disk. If the file in disk is corrupted, you will have to delete the file first.
    """
    if option == 'glove':
        destination = "glove.2024.wikigiga.50d.zip"
        dims = 50
        skip_first = False
    elif option == 'fasttext':
        destination = "wiki-news-300d-1M.vec.zip"
        dims = 300
        skip_first = True
    else:
        raise ValueError(f"Unkonwn option: {option}")

    if not Path(destination).is_file():
        if option == 'glove':
            !wget https://nlp.stanford.edu/data/wordvecs/glove.2024.wikigiga.50d.zip
        elif option == 'fasttext':
            !wget https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip


    return destination, dims, skip_first

In [4]:
import course_code.part2 as part2

# depends on internet connection (but say 5 minutes or less)
embtable_version, embtable_dims, embtable_metadata = download_pretrained('fasttext') # in case stanford.edu is down, use fasttext (but glove is preferred)

trained_emb_model = part2.clslib.PretrainedEmbeddings(embtable_version, embtable_dims, skip_first_line=embtable_metadata)

--2026-11-18 11:18:40--  https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 2600:9000:2062:4e00:13:6e38:acc0:93a1, 2600:9000:2062:3800:13:6e38:acc0:93a1, 2600:9000:2062:a00:13:6e38:acc0:93a1, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|2600:9000:2062:4e00:13:6e38:acc0:93a1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 681808098 (650M) [application/zip]
Saving to: ‘wiki-news-300d-1M.vec.zip’

wiki-news-300d-1M.v 100%[===================>] 650.22M  58.6MB/s    in 11s     

2026-11-18 11:18:51 (59.3 MB/s) - ‘wiki-news-300d-1M.vec.zip’ saved [681808098/681808098]



In [6]:
import nltk
nltk.download("wordnet")

[nltk_data] Downloading package wordnet to /home/owen/nltk_data...


True

In [10]:
import random
import numpy as np
from nltk.corpus import wordnet as wn   # pip install nltk; nltk.download("wordnet")
from sklearn.metrics import roc_auc_score

random.seed(0)

def cos(a, b):
    va, vb = trained_emb_model.embed_word(a), trained_emb_model.embed_word(b)
    return float(va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb) + 1e-9))

# --- Build gold synonyms: lemmas that share a synset ---
syns = {}                                 # word -> set of synonyms
for synset in wn.all_synsets():
    lemmas = {l.name().lower() for l in synset.lemmas()
              if "_" not in l.name() and l.name().lower() in trained_emb_model.vocab}
    for w in lemmas:
        syns.setdefault(w, set()).update(lemmas - {w})
syns = {w: s for w, s in syns.items() if s}

# --- Test 1: positive vs. random pairs ---
pos = [(w, s) for w, ss in syns.items() for s in ss if w < s]
words = trained_emb_model.vocab.words
neg = [(random.choice(words), random.choice(words)) for _ in range(len(pos))]
scores = [cos(a, b) for a, b in pos] + [cos(a, b) for a, b in neg]
labels = [1] * len(pos) + [0] * len(neg)
print("ROC-AUC:", roc_auc_score(labels, scores))
print("mean cos synonyms:", np.mean(scores[:len(pos)]),
      "| random:", np.mean(scores[len(pos):]))

ROC-AUC: 0.8336239576454529
mean cos synonyms: 0.47300897933656877 | random: 0.2729284106811012
